# Deep Learning based Anomaly Detection in Surveillance System
**Methodist College of Engineering and Technology**  
**Department of Computer Science and Engineering**  
**Batch:** PWCSE22B07  
**Guide:** Mrs. Deepthi Joshi, Associate Professor

---

## Fix: Wrong-Class False Positive

| Problem | Example from your test |
|---|---|
| Model detected `gun 0.72` | But you were holding a **knife** |
| Old code only flagged FP when `conf < threshold` | Wrong-class detections were silently counted as TP |
| **Fix** | Press **`F`** key in the video window to mark last detection as wrong-class FP |

### Key Controls During Live Detection
| Key | Action |
|---|---|
| `Q` | Quit and print session summary |
| `F` | Mark last detection as **False Positive** (wrong class) |
| `C` | Undo last FP mark (convert back to TP) |

## Cell 1 — Install Dependencies

In [ ]:
!pip install ultralytics opencv-python-headless

## Cell 2 — Imports

In [ ]:
import cv2
import csv
import os
import time
import tkinter as tk
from tkinter import messagebox
from ultralytics import YOLO
import winsound
from collections import defaultdict

## Cell 3 — Settings

In [ ]:
MODEL_PATH     = r"C:\Users\DELL\Downloads\best.pt"
CONF_THRESHOLD = 0.65
ALERT_COOLDOWN = 5
LOG_FILE       = "detection_log.csv"
TARGET_CLASSES = ["gun", "knife", "crowbar", "scissor"]

## Cell 4 — Load YOLOv8 Model

In [ ]:
model = YOLO(MODEL_PATH)
print("Model class mapping:", model.names)

## Cell 5 — Initialise Log File

In [ ]:
if not os.path.exists(LOG_FILE):
    with open(LOG_FILE, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([
            "Timestamp", "Detected_Object", "Confidence",
            "Image_File", "FP_WrongClass",
            "Precision", "Recall", "F1"
        ])
print(f"Log file ready: {LOG_FILE}")

## Cell 6 — Metrics State

| Type | Cause | How detected |
|---|---|---|
| **FP-LowConf** | Target class but `conf < threshold` | Automatic |
| **FP-WrongClass** | High conf but wrong class (knife shown as gun) | Press `F` key |
| **FN** | No detection at all | Proxy: frames with no target found |

In [ ]:
metrics = {
    "total_frames"          : 0,
    "frames_with_detection" : 0,
    "frames_no_detection"   : 0,
    "TP"                    : 0,
    "FP_low_conf"           : 0,
    "FP_wrong_class"        : 0,
    "total_detections"      : 0,
    "conf_sum"              : 0.0,
    "conf_count"            : 0,
    "class_tp"              : defaultdict(int),
    "class_fp"              : defaultdict(int),
    "class_fp_wrong"        : defaultdict(int),
    "class_conf_sum"        : defaultdict(float),
    "alert_count"           : 0,
    "wrong_class_alerts"    : 0,
    "last_detected_class"   : None,
    "last_detection_time"   : 0,
    "fps_history"           : [],
    "session_start"         : time.time(),
}
print("Metrics state initialised.")

## Cell 7 — Precision / Recall / F1 Helper

In [ ]:
def compute_prf(tp, fp, fn):
    precision = tp / (tp + fp)   if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn)   if (tp + fn) > 0 else 0.0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)
    return precision, recall, f1

def total_fp(m):
    return m["FP_low_conf"] + m["FP_wrong_class"]

## Cell 8 — Alert Function

In [ ]:
def show_alert(object_name):
    root = tk.Tk()
    root.withdraw()
    messagebox.showwarning(
        "SECURITY ALERT",
        f"Detected: {object_name.upper()}"
    )
    root.destroy()

## Cell 9 — On-Screen Metrics Overlay

In [ ]:
def draw_metrics_overlay(frame, fps, metrics, fp_flash=False):
    overlay = frame.copy()
    tp  = metrics["TP"]
    fp  = total_fp(metrics)
    fn  = metrics["frames_no_detection"]
    precision, recall, f1 = compute_prf(tp, fp, fn)
    accuracy = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0
    avg_conf = (metrics["conf_sum"] / metrics["conf_count"] * 100
                if metrics["conf_count"] > 0 else 0.0)
    det_rate = (metrics["frames_with_detection"] / metrics["total_frames"] * 100
                if metrics["total_frames"] > 0 else 0.0)
    elapsed  = time.strftime("%H:%M:%S",
                             time.gmtime(time.time() - metrics["session_start"]))
    panel_h = 310
    cv2.rectangle(overlay, (8, 8), (320, 8 + panel_h), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.50, frame, 0.50, 0, frame)
    font  = cv2.FONT_HERSHEY_SIMPLEX
    WHITE = (230, 230, 230)
    GOLD  = (80,  200, 255)
    GREEN = (100, 230, 100)
    CYAN  = (220, 220,  80)
    RED   = ( 80,  80, 255)
    def put(text, row, color=WHITE, scale=0.50, weight=1):
        cv2.putText(frame, text, (18, 30 + row * 24),
                    font, scale, color, weight, cv2.LINE_AA)
    put("SURVEILLANCE METRICS",             0, GOLD,  0.54, 2)
    put(f"FPS          : {fps:>6.1f}",      1, GREEN)
    put(f"Session      : {elapsed}",         2, WHITE)
    put(f"Frames       : {metrics['total_frames']:>6}",         3, WHITE)
    put(f"Detection %  : {det_rate:>5.1f}%",                    4, WHITE)
    put(f"Avg Conf     : {avg_conf:>5.1f}%",                    5, WHITE)
    put(f"Alerts Fired : {metrics['alert_count']:>6}",          6, CYAN)
    put("--------------------------------",   7, (80, 80, 80))
    put("PERFORMANCE EVALUATION",            8, GOLD,  0.50, 2)
    put(f"Accuracy     : {accuracy*100:>5.1f}%",   9,  GREEN)
    put(f"Precision    : {precision*100:>5.1f}%",  10, GREEN)
    put(f"Recall       : {recall*100:>5.1f}%",     11, GREEN)
    put(f"F1 Score     : {f1*100:>5.1f}%",         12, GREEN)
    put(f"TP           : {tp}",                     13, GREEN)
    put(f"FP-LowConf   : {metrics['FP_low_conf']}",
        14, RED if fp_flash else WHITE)
    put(f"FP-WrongClass: {metrics['FP_wrong_class']}  [press F]",
        15, RED if (fp_flash or metrics['FP_wrong_class'] > 0) else WHITE)
    put(f"FN (proxy)   : {fn}",              16, WHITE, 0.44)
    put("[F]=mark FP  [C]=undo  [Q]=quit",   17, (160, 160, 160), 0.40)
    if metrics["class_tp"] or metrics["class_fp_wrong"]:
        base_y = 8 + panel_h + 12
        cv2.putText(frame, "PER-CLASS BREAKDOWN",
                    (18, base_y), font, 0.48, GOLD, 1, cv2.LINE_AA)
        row_y = base_y + 20
        all_cls = set(list(metrics["class_tp"].keys()) +
                      list(metrics["class_fp"].keys()))
        for cls in sorted(all_cls,
                          key=lambda x: metrics["class_tp"][x], reverse=True):
            tp_c  = metrics["class_tp"][cls]
            fp_c  = metrics["class_fp"][cls]
            fpw_c = metrics["class_fp_wrong"][cls]
            total = tp_c + fp_c
            avg_c = (metrics["class_conf_sum"][cls] / total * 100
                     if total > 0 else 0.0)
            prec_c, _, _ = compute_prf(tp_c, fp_c, 0)
            label = (f"  {cls:<9}: TP={tp_c} FP={fp_c}(W:{fpw_c}) "
                     f"P={prec_c*100:.0f}% C={avg_c:.0f}%")
            color = RED if fpw_c > 0 else WHITE
            cv2.putText(frame, label, (18, row_y),
                        font, 0.40, color, 1, cv2.LINE_AA)
            row_y += 18
    return frame

## Cell 10 — Session Summary

In [ ]:
def print_summary(metrics):
    tp  = metrics["TP"]
    fp  = total_fp(metrics)
    fn  = metrics["frames_no_detection"]
    precision, recall, f1 = compute_prf(tp, fp, fn)
    accuracy = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0
    avg_fps  = (sum(metrics["fps_history"]) / len(metrics["fps_history"])
                if metrics["fps_history"] else 0.0)
    avg_conf = (metrics["conf_sum"] / metrics["conf_count"] * 100
                if metrics["conf_count"] > 0 else 0.0)
    elapsed  = time.time() - metrics["session_start"]
    sep = "=" * 60
    print(f"\n{sep}")
    print("   Deep Learning Anomaly Detection - Batch PWCSE22B07")
    print("   SESSION PERFORMANCE REPORT")
    print(sep)
    print(f"  Session duration         : {elapsed:.1f} sec")
    print(f"  Total frames processed   : {metrics['total_frames']}")
    print(f"  Average FPS              : {avg_fps:.2f}")
    print(f"  Avg detection confidence : {avg_conf:.2f}%")
    print(f"  Total alerts fired       : {metrics['alert_count']}")
    print(f"  Wrong-class alerts (FP)  : {metrics['wrong_class_alerts']}")
    print("-" * 60)
    print("  PERFORMANCE EVALUATION  (PPT Slide 28-H)")
    print(f"  Accuracy                 : {accuracy*100:.2f}%")
    print(f"  Precision                : {precision*100:.2f}%")
    print(f"  Recall                   : {recall*100:.2f}%")
    print(f"  F1 Score                 : {f1*100:.2f}%")
    print(f"  True  Positives   (TP)   : {tp}")
    print(f"  False Pos-LowConf (FP1)  : {metrics['FP_low_conf']}")
    print(f"  False Pos-WrongCls(FP2)  : {metrics['FP_wrong_class']}  <- knife detected as gun")
    print(f"  False Negatives   (FN)   : {fn}  [proxy: frames w/ no detection]")
    print("-" * 60)
    print("  PER-CLASS BREAKDOWN:")
    all_cls = set(list(metrics["class_tp"].keys()) +
                  list(metrics["class_fp"].keys()))
    if all_cls:
        for cls in sorted(all_cls,
                          key=lambda x: metrics["class_tp"][x], reverse=True):
            tp_c  = metrics["class_tp"][cls]
            fp_c  = metrics["class_fp"][cls]
            fpw_c = metrics["class_fp_wrong"][cls]
            total = tp_c + fp_c
            avg_c = (metrics["class_conf_sum"][cls] / total * 100
                     if total > 0 else 0.0)
            prec_c, _, _ = compute_prf(tp_c, fp_c, 0)
            wrong_note = f"  *** {fpw_c} wrong-class FP ***" if fpw_c > 0 else ""
            print(f"    {cls:<10} | TP={tp_c:<3} FP={fp_c:<3}(wrong={fpw_c}) "
                  f"Prec={prec_c*100:5.1f}%  AvgConf={avg_c:5.1f}%{wrong_note}")
    else:
        print("    No target-class detections this session.")
    print(sep + "\n")

## Cell 11 — Main Detection Loop

| Key | Action |
|---|---|
| `Q` | Quit |
| `F` | Mark last detection as wrong-class FP (e.g. knife shown as gun) |
| `C` | Undo last FP mark |

In [ ]:
cap             = cv2.VideoCapture(0)
last_alert_time = 0
prev_time       = time.time()
fp_flash_until  = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    now = time.time()
    fps = 1.0 / max(now - prev_time, 1e-9)
    prev_time = now
    metrics["fps_history"].append(fps)
    metrics["total_frames"] += 1

    results         = model(frame, conf=CONF_THRESHOLD)
    annotated_frame = results[0].plot()
    frame_triggered = False

    for box in results[0].boxes:
        class_id   = int(box.cls[0])
        class_name = model.names[class_id]
        confidence = float(box.conf[0])
        print(f"Detected: {class_name}  |  Conf: {confidence:.2f}")
        metrics["total_detections"] += 1
        metrics["conf_sum"]         += confidence
        metrics["conf_count"]       += 1

        if class_name in TARGET_CLASSES:
            if confidence >= CONF_THRESHOLD:
                metrics["TP"] += 1
                metrics["class_tp"][class_name]       += 1
                metrics["class_conf_sum"][class_name] += confidence
                frame_triggered = True
                metrics["last_detected_class"] = class_name
                metrics["last_detection_time"] = now

                if now - last_alert_time > ALERT_COOLDOWN:
                    timestamp  = time.strftime("%Y-%m-%d %H:%M:%S")
                    image_name = f"alert_{int(now)}.jpg"
                    cv2.imwrite(image_name, annotated_frame)
                    tp_ = metrics["TP"]
                    fp_ = total_fp(metrics)
                    fn_ = metrics["frames_no_detection"]
                    p_, r_, f_ = compute_prf(tp_, fp_, fn_)
                    with open(LOG_FILE, mode='a', newline='') as lf:
                        csv.writer(lf).writerow([
                            timestamp, class_name,
                            f"{confidence:.4f}", image_name,
                            "pending",
                            f"{p_:.4f}", f"{r_:.4f}", f"{f_:.4f}"
                        ])
                    winsound.Beep(1000, 800)
                    show_alert(class_name)
                    metrics["alert_count"] += 1
                    last_alert_time = now
            else:
                metrics["FP_low_conf"] += 1
                metrics["class_fp"][class_name]       += 1
                metrics["class_conf_sum"][class_name] += confidence

    if not frame_triggered:
        metrics["frames_no_detection"] += 1
    else:
        metrics["frames_with_detection"] += 1

    fp_flash        = now < fp_flash_until
    annotated_frame = draw_metrics_overlay(annotated_frame, fps, metrics, fp_flash)
    cv2.imshow("Deep Learning Anomaly Detection - Surveillance System",
               annotated_frame)

    key = cv2.waitKey(1) & 0xFF

    if key == ord('q'):
        break

    elif key == ord('f'):
        cls = metrics["last_detected_class"]
        if cls and (now - metrics["last_detection_time"]) < 10:
            if metrics["class_tp"][cls] > 0:
                metrics["class_tp"][cls] -= 1
                metrics["TP"]            -= 1
            metrics["FP_wrong_class"]      += 1
            metrics["class_fp"][cls]       += 1
            metrics["class_fp_wrong"][cls] += 1
            metrics["wrong_class_alerts"]  += 1
            fp_flash_until = now + 1.5
            print(f"[F KEY] Marked '{cls}' as WRONG-CLASS FP")
        else:
            print("[F KEY] No recent detection to mark")

    elif key == ord('c'):
        cls = metrics["last_detected_class"]
        if cls and metrics["class_fp_wrong"][cls] > 0:
            metrics["FP_wrong_class"]      -= 1
            metrics["class_fp"][cls]       -= 1
            metrics["class_fp_wrong"][cls] -= 1
            metrics["TP"]                  += 1
            metrics["class_tp"][cls]       += 1
            if metrics["wrong_class_alerts"] > 0:
                metrics["wrong_class_alerts"] -= 1
            print(f"[C KEY] Undone - '{cls}' converted back to TP")
        else:
            print("[C KEY] Nothing to undo")

cap.release()
cv2.destroyAllWindows()

## Cell 12 — Print Full Session Report

In [ ]:
print_summary(metrics)